# CS229 L03 — Probabilistic Interpretation, Logistic Regression & Newton's Method

**Stanford CS229 · Spring 2026 · Instructor: Chris**  
[▶ Lecture Video](https://www.youtube.com/watch?v=uJF_gL3jhxI) · [Official Notes](https://cs229.stanford.edu/notes/cs229-notes1.pdf) · [Course Website](https://cs229.stanford.edu/)

---

> 📌 *Lecture:* — instructor's exact words from the transcript  
> 🎯 **Interview:** — Q&A blocks for interview articulation

---

## 1. Why Probabilistic Interpretation?

> 📌 *Lecture:* "We're going to take something we already know from last time and give it a probabilistic interpretation. The hope is that this piece of modeling — just that piece — is changing. Once we get to the form of having a probabilistic model, that's going to allow us to extend it to many new and potentially unfamiliar settings. This maximum likelihood framework is really foundational — it's the bedrock."

**Last lecture:** We minimized $J(\theta) = \frac{1}{2}\sum_i (\theta^T x^{(i)} - y^{(i)})^2$ — but *why squared?*

**This lecture:** We derive that squared loss falls out naturally from a probabilistic assumption. This unlocks a general recipe for deriving loss functions for *any* problem.

> 📌 *Lecture:* "This principle of maximum likelihood is more general than least squares. As you come to a new problem, you want to model it. You can turn a crank and always get out some optimization procedure. That's going to allow us to solve a range of different things that in the machine learning literature look quite distinct, but are actually instances of one generalized framework."

**The MLE recipe:**
```
1. Write a probabilistic model p(y | x; θ)
2. Write the likelihood L(θ) = ∏ᵢ p(y⁽ⁱ⁾ | x⁽ⁱ⁾; θ)  [IID assumption]
3. Take log → log-likelihood ℓ(θ) = ∑ᵢ log p(y⁽ⁱ⁾ | x⁽ⁱ⁾; θ)
4. Maximize ℓ(θ) ↔ minimize -ℓ(θ)
5. Run gradient descent
```

> 🎯 **Interview:** *What is maximum likelihood estimation?*  
> MLE asks: among all parameter settings θ, which one makes the observed data most probable? We write the likelihood $L(\theta) = \prod_i p(y^{(i)} | x^{(i)}; \theta)$ — the probability of observing all the training labels given the inputs. We maximize this over θ. In practice we maximize the log-likelihood instead (log turns products into sums, which is numerically stable and easier to differentiate). The key insight: the choice of probabilistic model determines the loss function. Gaussian noise → squared loss. Bernoulli → cross-entropy. This is why MLE is so general.

## 2. MLE Derivation of Least Squares

### 2.1 The Generative Model

> 📌 *Lecture:* "We're going to assume the data is generated according to some process, and that we know something about that process. There exists a true theta out in the world. We don't get to see theta, but it exists. And then when we see yᵢ, some noise term εᵢ is introduced. That εᵢ captures what we're willing to model up to."

$$y^{(i)} = \theta^T x^{(i)} + \epsilon^{(i)}$$

**Assumptions on noise $\epsilon^{(i)}$:**

> 📌 *Lecture:* "What properties are we going to assume about the noise? We'll start with one of the simplest, which underlies a lot of what we do. First: the expected value of epsilon is zero — the error is not systematically biased. Second: they are IID — independent and identically distributed. This is a very strong assumption, but it's a good modeling assumption. It allows us to do a bunch of math."

1. $\mathbb{E}[\epsilon^{(i)}] = 0$ — zero mean, no systematic bias
2. $\text{Var}(\epsilon^{(i)}) = \sigma^2$ — fixed variance
3. **IID**: $\epsilon^{(1)}, \epsilon^{(2)}, ..., \epsilon^{(n)}$ are independent and identically distributed

With these three assumptions, the noise is Gaussian:
$$\epsilon^{(i)} \sim \mathcal{N}(0, \sigma^2)$$

> 📌 *Lecture:* "Gaussians are uniquely defined if you make this assumption — zero mean and just the variance. They show up everywhere in statistical analysis because they're the workhorse. And they're a really good first approximation. The Gaussian isn't a categorical kind of thing you're worried about — you want to know if it's good enough for whatever you care about."

### 2.2 The Gaussian PDF

$$p(z; \mu, \sigma^2) = \frac{1}{\sqrt{2\pi}\sigma} \exp\left(-\frac{(z-\mu)^2}{2\sigma^2}\right)$$

> 📌 *Lecture:* "This is a probability distribution. The normalization constant is probably the least important part of the formula. If you forget this, you can do almost all machine learning — constants will fall away when we start to minimize. The action goes on under this exponential. This says my distance to the mean squared is going to enter for my distribution — how likely it is roughly."

| Symbol | Meaning |
|---|---|
| $\mu$ | Mean (center of the bell curve) |
| $\sigma^2$ | Variance (width² of the bell curve) |
| $\sigma$ | Standard deviation (width) |
| $\frac{1}{\sqrt{2\pi}\sigma}$ | Normalization constant (area under curve = 1) |

### 2.3 The Conditional Distribution

Since $y^{(i)} = \theta^T x^{(i)} + \epsilon^{(i)}$ and $\epsilon^{(i)} \sim \mathcal{N}(0, \sigma^2)$:

$$y^{(i)} | x^{(i)}; \theta \sim \mathcal{N}(\theta^T x^{(i)}, \sigma^2)$$

The mean shifts to $\theta^T x^{(i)}$, the variance stays $\sigma^2$.

### 2.4 The Likelihood

By the IID assumption, the joint likelihood factors:
$$L(\theta) = \prod_{i=1}^{n} p(y^{(i)} | x^{(i)}; \theta) = \prod_{i=1}^{n} \frac{1}{\sqrt{2\pi}\sigma} \exp\left(-\frac{(y^{(i)} - \theta^T x^{(i)})^2}{2\sigma^2}\right)$$

> 📌 *Lecture:* "Why do we take this product over all the data? Because of IID. The probability of all the x's and y's simultaneously can be represented as a product. IID is allowing me to say that gigantic function can be represented as a product. That's what independence means — it really is a factorization."

### 2.5 The Log-Likelihood

> 📌 *Lecture:* "We take logs. We love logarithms. The reason is they're going to turn this nasty product — which is numerically unstable — into some kind of nice closed-form looking sum. We get the log of the normalization constants minus all these characters."

$$\ell(\theta) = \log L(\theta) = -n \log(\sqrt{2\pi}\sigma) - \frac{1}{2\sigma^2} \sum_{i=1}^{n}(y^{(i)} - \theta^T x^{(i)})^2$$

### 2.6 MLE = Least Squares

Maximizing $\ell(\theta)$ w.r.t. $\theta$:
- The $-n\log(\sqrt{2\pi}\sigma)$ term doesn't involve $\theta$ → drop it
- $-\frac{1}{2\sigma^2}$ is a negative constant → maximizing is the same as minimizing:

$$\arg\max_\theta \ell(\theta) = \arg\min_\theta \frac{1}{2}\sum_{i=1}^{n}(y^{(i)} - \theta^T x^{(i)})^2 = \arg\min_\theta J(\theta)$$

$$\boxed{\text{MLE under Gaussian noise} = \text{Least Squares}}$$

> 📌 *Lecture:* "What we've justified is that least squares is basically this model where we have fixed variance, but we have errors that are IID, mean zero. That's it. So, this is just a much more principled way of motivating least squares."

> 🎯 **Interview:** *Why does least squares loss correspond to a Gaussian noise assumption?*  
> Assume the true labels follow $y = \theta^Tx + \epsilon$ where $\epsilon \sim \mathcal{N}(0, \sigma^2)$. The conditional distribution of $y$ given $x$ is then $\mathcal{N}(\theta^Tx, \sigma^2)$. The log-likelihood of the training data under IID sampling is $\ell(\theta) = -\frac{1}{2\sigma^2}\sum_i (y^{(i)} - \theta^Tx^{(i)})^2 + \text{const}$. Maximizing this is equivalent to minimizing the squared loss. So squared loss is not arbitrary — it's what you get when you assume Gaussian noise. Similarly, L1 loss corresponds to Laplace noise, and Poisson loss corresponds to count data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Visualize the Gaussian noise model
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Gaussian PDF shapes
x = np.linspace(-4, 4, 300)
for sigma, color in [(0.5, 'steelblue'), (1.0, 'coral'), (2.0, 'green')]:
    axes[0].plot(x, norm.pdf(x, 0, sigma), label=f'σ={sigma}', color=color)
axes[0].set_title('Gaussian PDF: N(0, σ²)')
axes[0].set_xlabel('ε (noise)')
axes[0].set_ylabel('p(ε)')
axes[0].legend()
axes[0].axvline(0, color='black', linestyle='--', alpha=0.3)

# Right: regression with noise
np.random.seed(42)
n = 60
x_data = np.linspace(0, 3, n)
theta_true = np.array([1.0, 2.0])  # [bias, slope]
sigma = 0.5
y_data = theta_true[0] + theta_true[1] * x_data + np.random.randn(n) * sigma

axes[1].scatter(x_data, y_data, alpha=0.5, label='Training data (y = θᵀx + ε)')
axes[1].plot(x_data, theta_true[0] + theta_true[1]*x_data, 'r-', label='True line θᵀx')

# Show noise distributions at a few x points
for x0 in [0.5, 1.5, 2.5]:
    y0 = theta_true[0] + theta_true[1] * x0
    eps = np.linspace(-1.5, 1.5, 100)
    axes[1].plot(x0 + norm.pdf(eps, 0, sigma)*0.3, y0 + eps, 'gray', alpha=0.4)

axes[1].set_title('y | x; θ ~ N(θᵀx, σ²)')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].legend()
plt.tight_layout()
plt.show()
print("Each y is the true θᵀx plus Gaussian noise.")
print("MLE finds θ that maximizes P(y|x) = minimizes squared loss.")

## 3. Logistic Regression — Binary Classification

### 3.1 Why Not Least Squares for Classification?

> 📌 *Lecture:* "Why do I need to do anything different? I'll take my data, my x's and y's, why don't I just run them through the least squares thing? What does that kind of look like? Let's draw a picture and see the traditional pitfall. What happens is as you start to get X's that are going farther and farther away, instead of finding the exact spot that splits them, I'm somehow trying to find their center of mass. That seems inappropriate when I'm just trying to find a classification decision surface."

**The problem:** Least squares on binary labels is sensitive to outliers — distant points in the correct class can pull the decision boundary the wrong way.

> 📌 *Lecture:* "All that said, if you have actual data and you try this out, least squares will work more often than you think. High-dimensional data is pretty weird — your closest point and farthest point are often the same distance. Very strange things happen in high dimensions. So sometimes you just run a model that's pretty bad but does a good enough job."

**The fix:** Use a model designed for binary outputs — logistic regression.

### 3.2 The Sigmoid (Logistic) Function

> 📌 *Lecture:* "We're going to have these things called link functions. The hypothesis is still going to be linear, but we're going to add in a non-linear element G. And G is going to be called a link function. What would you want G to have? Probably be smooth and monotone. There are many that you can pick, but here's one that's very, very popular."

$$g(z) = \sigma(z) = \frac{1}{1 + e^{-z}}$$

The logistic regression hypothesis:
$$h_\theta(x) = g(\theta^T x) = \frac{1}{1 + e^{-\theta^T x}}$$

**Properties:**
- $g(z) \in (0, 1)$ for all $z \in \mathbb{R}$ — always a valid probability
- $g(z) \to 1$ as $z \to +\infty$; $g(z) \to 0$ as $z \to -\infty$
- $g(0) = 0.5$ — at the decision boundary
- Smooth and monotone increasing — differentiable everywhere
- Derivative: $g'(z) = g(z)(1 - g(z))$ — elegant and useful for backprop

> 📌 *Lecture:* "Why don't we just use a threshold — a step function? The reason is it's non-differentiable and so it's not convenient mathematically. We just focus on this nice one because it has this beautiful scaling between zero and one. You'll sometimes hear people call these logits — that's just a fancy word for $\theta^Tx$ before the sigmoid."

**Probabilistic interpretation:**
$$h_\theta(x) = P(y=1 | x; \theta)$$
$$1 - h_\theta(x) = P(y=0 | x; \theta)$$

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    g = sigmoid(z)
    return g * (1 - g)  # elegant form: g'(z) = g(z)(1-g(z))

z = np.linspace(-6, 6, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(z, sigmoid(z), 'steelblue', lw=2, label='σ(z) = 1/(1+e⁻ᶻ)')
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='σ(0) = 0.5')
axes[0].axhline(1.0, color='red', linestyle='--', alpha=0.3, label='Upper bound = 1')
axes[0].axhline(0.0, color='green', linestyle='--', alpha=0.3, label='Lower bound = 0')
axes[0].set_xlabel('z = θᵀx  (logit)')
axes[0].set_ylabel('σ(z) = P(y=1|x; θ)')
axes[0].set_title('Sigmoid Function')
axes[0].legend(fontsize=9)

axes[1].plot(z, sigmoid_derivative(z), 'coral', lw=2, label="σ'(z) = σ(z)(1-σ(z))")
axes[1].set_xlabel('z')
axes[1].set_ylabel("σ'(z)")
axes[1].set_title("Sigmoid Derivative — g'(z) = g(z)(1-g(z))")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"σ(0) = {sigmoid(0):.3f}  (decision boundary)")
print(f"σ(2) = {sigmoid(2):.3f}  (confident positive)")
print(f"σ(-2) = {sigmoid(-2):.3f} (confident negative)")
print(f"Max derivative at z=0: {sigmoid_derivative(0):.3f}")

### 3.3 MLE Derivation of Cross-Entropy Loss

**Probabilistic model:** label $y \in \{0, 1\}$ follows a Bernoulli distribution:
$$P(y=1 | x; \theta) = h_\theta(x), \quad P(y=0 | x; \theta) = 1 - h_\theta(x)$$

Written compactly (the Bernoulli likelihood trick):
$$p(y | x; \theta) = h_\theta(x)^y \cdot (1 - h_\theta(x))^{1-y}$$

> 📌 *Lecture:* "This notation is just an if-then written in a clever stat-sy machine learning way. It says either yᵢ is zero and this term is active, or yᵢ is one and this term. When they're off, they return one. Just convince yourself that this function does what you think it does."

**Likelihood** (IID):
$$L(\theta) = \prod_{i=1}^{n} h_\theta(x^{(i)})^{y^{(i)}} (1 - h_\theta(x^{(i)}))^{1-y^{(i)}}$$

**Log-likelihood:**
$$\ell(\theta) = \sum_{i=1}^{n} \left[ y^{(i)} \log h_\theta(x^{(i)}) + (1-y^{(i)}) \log(1 - h_\theta(x^{(i)})) \right]$$

**Loss** (negate to minimize):
$$\boxed{J(\theta) = -\frac{1}{n}\sum_{i=1}^{n} \left[ y^{(i)} \log h_\theta(x^{(i)}) + (1-y^{(i)}) \log(1 - h_\theta(x^{(i)})) \right]}$$

This is **binary cross-entropy loss** — derived purely from MLE under a Bernoulli model.

$$\boxed{\text{MLE under Bernoulli} = \text{Binary Cross-Entropy}}$$

> 📌 *Lecture:* "Once we have a forward model, we basically turn a crank and always get out some optimization procedure. The model, log space, additive sum, gradient descent. We try to figure out how likely our data set is, go to log space — a nice monotone mapping — and boom, we're off to solve this in a big additive sum. It just snaps together."

**The gradient — error × feature again:**
$$\frac{\partial}{\partial \theta_j} J(\theta) = \frac{1}{n}\sum_{i=1}^{n}(h_\theta(x^{(i)}) - y^{(i)}) x_j^{(i)}$$

> 📌 *Lecture:* "The remarkable thing is you can still get to something that has this nice familiar form where you measure the error and multiply by xⱼ. This form — error and correct form — is very, very common. All the nastiness is put into the point difference, and then you do an update. It's quite remarkable."

**The gradient has the same pattern as linear regression** — the only difference is $h_\theta(x)$ is now sigmoid instead of $\theta^Tx$.

> 🎯 **Interview:** *Derive the cross-entropy loss for logistic regression from first principles.*  
> Model: $P(y|x;\theta) = h_\theta(x)^y (1-h_\theta(x))^{1-y}$ where $h_\theta(x) = \sigma(\theta^Tx)$. Log-likelihood: $\ell(\theta) = \sum_i [y^{(i)} \log h^{(i)} + (1-y^{(i)}) \log(1-h^{(i)})]$. Negate to get the loss. The gradient works out cleanly to $(h_\theta(x^{(i)}) - y^{(i)}) x_j^{(i)}$ — same error × feature pattern as least squares, just with sigmoid instead of linear prediction.

In [ ]:
def binary_cross_entropy(h, y):
    eps = 1e-9  # numerical stability
    return -np.mean(y * np.log(h + eps) + (1 - y) * np.log(1 - h + eps))

def logistic_regression_gd(X, y, alpha=0.1, n_iter=1000):
    n, d = X.shape
    theta = np.zeros(d)
    losses = []
    for _ in range(n_iter):
        h = sigmoid(X @ theta)                    # predictions
        error = h - y                              # (h - y)
        grad = X.T @ error / n                    # error × feature
        theta -= alpha * grad
        losses.append(binary_cross_entropy(h, y))
    return theta, losses

# Generate binary classification data
np.random.seed(42)
n = 200
X_pos = np.random.randn(n//2, 2) + [2, 2]
X_neg = np.random.randn(n//2, 2) + [-1, -1]
X_cls = np.vstack([X_pos, X_neg])
y_cls = np.array([1]*(n//2) + [0]*(n//2))

# Add bias column
X_cls_bias = np.hstack([np.ones((n, 1)), X_cls])

theta_lr, losses_lr = logistic_regression_gd(X_cls_bias, y_cls, alpha=0.5, n_iter=500)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Decision boundary
xx, yy = np.meshgrid(np.linspace(-4, 6, 200), np.linspace(-4, 6, 200))
XY = np.c_[np.ones(xx.ravel().shape), xx.ravel(), yy.ravel()]
Z = sigmoid(XY @ theta_lr).reshape(xx.shape)
axes[0].contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.3)
axes[0].contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
axes[0].scatter(X_pos[:, 0], X_pos[:, 1], c='steelblue', label='y=1', alpha=0.6)
axes[0].scatter(X_neg[:, 0], X_neg[:, 1], c='coral', label='y=0', alpha=0.6)
axes[0].set_title('Logistic Regression Decision Boundary')
axes[0].legend()

# Loss curve
axes[1].plot(losses_lr)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Binary Cross-Entropy Loss')
axes[1].set_title('Cross-Entropy Loss During Training')

plt.tight_layout()
plt.show()

# Accuracy
h_final = sigmoid(X_cls_bias @ theta_lr)
acc = np.mean((h_final >= 0.5) == y_cls)
print(f"Final loss: {losses_lr[-1]:.4f}")
print(f"Accuracy: {acc*100:.1f}%")
print(f"Gradient pattern: (h - y) × x  [same as linear regression]")

## 4. Newton's Method

> 📌 *Lecture:* "I'm going to show you Newton's method just very briefly. It's historically important and an interesting algorithm in its own right. This is a simple method if you haven't seen it before."

**Idea:** Instead of only using first-order information (gradient = slope), use second-order information (Hessian = curvature) to determine how far to step.

**Scalar case (finding zero of f):**

> 📌 *Lecture:* "We have some function we want to find the zero of. We start with an initial guess theta zero. We compute the gradient and use this local linear model — this line — to find where it crosses zero. That tells us exactly how far to go. It doesn't have a step size, which is quite a nice property."

Taylor approximation at $\theta_t$:
$$f(\theta) \approx f(\theta_t) + f'(\theta_t)(\theta - \theta_t)$$

Set $= 0$ and solve for $\theta$:
$$\theta_{t+1} = \theta_t - \frac{f(\theta_t)}{f'(\theta_t)}$$

**For optimization** (finding zero of the gradient $\nabla J$):
$$\theta_{t+1} = \theta_t - \frac{\nabla_\theta J(\theta_t)}{\nabla^2_\theta J(\theta_t)}$$

**Vector case** (replace $f''$ with the Hessian $H$):
$$\theta_{t+1} = \theta_t - H^{-1} \nabla_\theta J(\theta_t)$$

where $H_{jk} = \frac{\partial^2 J}{\partial \theta_j \partial \theta_k}$

> 📌 *Lecture:* "The Hessian isn't something mysterious. It's just saying: if there's a really big difference in this direction versus a small one in that direction, I should probably step more in the directions I care about. That's roughly all that's going on."

### Why Not Newton for Large-Scale ML?

> 📌 *Lecture:* "SGD has O(D) compute per step. Newton is N × D² + D³. The D cubed is coming from the inversions. You do very few steps — statistically it's extremely efficient. But each of those steps is extremely, extremely expensive. Mini-batch SGD is the workhorse of ML."

| Algorithm | Per-step cost | Steps to converge | Used when |
|---|---|---|---|
| SGD | $O(BD)$ | Many | Always in DL — large $n$, $d$ |
| Batch GD | $O(nD)$ | Moderate | Small datasets |
| Newton | $O(nD^2 + D^3)$ | Few | Stats packages, small $d$ |
| L-BFGS | $O(nD)$ | Moderate | Stats, when curvature matters |

> 📌 *Lecture:* "Newton's method is used all over the place in classical statistics. If you're going to train a model in a stats package with, you know, 20 features, you'd use Newton or L-BFGS. But in machine learning, N is absolutely massive, and so is D. That's a very weird space. A social scientist would never in a million years use SGD — it'd be too noisy, too hard to tune. Newton's method doesn't have step sizes — just press the button and it works. So it does get used quite a bit more in those kinds of problems."

**The Adam connection:**

> 📌 *Lecture:* "Adam is a combination of those ideas [diagonal Hessian approximation] with what's called momentum. AdaGrad was trying to estimate a little bit of what's called curvature information — how much should I trust a step in every direction. The Hessian tells you that. If you assume H is diagonal, that's super cheap. That's what AdaGrad was doing. Those ideas form the basis of Adam."

$$\text{Adam} \approx \text{SGD} + \text{momentum} + \text{diagonal curvature estimate}$$

> 🎯 **Interview:** *Why does deep learning use SGD/Adam instead of Newton's method?*  
> Newton's method requires computing and inverting the Hessian — $O(D^3)$ per step where $D$ is the number of parameters. For a 70B parameter model, this is completely infeasible. SGD requires only $O(D)$ per step (one gradient computation), making it the only tractable algorithm at scale. Adam approximates second-order information cheaply by maintaining diagonal estimates of the gradient's second moment (adaptive per-parameter learning rates) — capturing the spirit of Newton's method without the $O(D^3)$ cost. The tradeoff: Newton converges in very few steps but each step is expensive; SGD takes many noisy steps but each is cheap.

In [ ]:
# Newton's method vs GD on a simple convex function
def f(x): return x**4 - 3*x**3 + 2  # example function
def f_prime(x): return 4*x**3 - 9*x**2
def f_double_prime(x): return 12*x**2 - 18*x

# Newton's method: find zero of f_prime (i.e., minimum of f)
x = 3.0  # starting point
newton_path = [x]
for _ in range(8):
    x = x - f_prime(x) / f_double_prime(x)
    newton_path.append(x)

# Gradient descent with fixed step size
x_gd = 3.0
gd_path = [x_gd]
for _ in range(50):
    x_gd = x_gd - 0.02 * f_prime(x_gd)
    gd_path.append(x_gd)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x_plot = np.linspace(-0.5, 3.5, 300)
for ax, path, label, color, title in [
    (axes[0], newton_path, "Newton's method", 'steelblue', f"Newton's Method ({len(newton_path)-1} steps)"),
    (axes[1], gd_path, 'Gradient Descent', 'coral', f'Gradient Descent ({len(gd_path)-1} steps)')
]:
    ax.plot(x_plot, f(x_plot), 'k-', lw=2)
    ax.scatter(path, f(np.array(path)), c=color, zorder=5, s=50)
    ax.plot(path, f(np.array(path)), '--', color=color, alpha=0.5)
    ax.set_xlabel('θ')
    ax.set_ylabel('f(θ)')
    ax.set_title(title)
    ax.set_ylim(-3, 5)

plt.tight_layout()
plt.show()
print(f"Newton converged in {len(newton_path)-1} steps.")
print(f"GD took {len(gd_path)-1} steps for similar convergence.")
print(f"But Newton's each step costs O(D³) — prohibitive for large D.")

## 5. The MLE Framework — Summary

> 📌 *Lecture:* "That flow — the thing I want you to take away from all of this — is this maximum likelihood method. You set up a probabilistic model of how you think the world behaves, and that determines what the parameters you're solving are. That's the thing that's going to generalize across various different setups — from continuous to discrete to all kinds of interesting new models, all the way into the modern AI models we use today."

| Model | Noise assumption | Loss function |
|---|---|---|
| Linear regression | $\epsilon \sim \mathcal{N}(0, \sigma^2)$ | Squared loss: $(y - \hat{y})^2$ |
| Logistic regression | $y \sim \text{Bernoulli}(h_\theta(x))$ | Binary cross-entropy |
| Poisson regression | $y \sim \text{Poisson}(\lambda)$ | Poisson log-loss |
| Softmax regression | $y \sim \text{Categorical}(\text{softmax}(z))$ | Cross-entropy (multiclass) |

**The universal recipe:**
1. Choose distribution for $p(y | x; \theta)$
2. Compute $\ell(\theta) = \sum_i \log p(y^{(i)} | x^{(i)}; \theta)$
3. Minimize $-\ell(\theta)$ with SGD

> 🎯 **Interview:** *What is the relationship between MLE and loss functions in machine learning?*  
> Every standard loss function in ML corresponds to MLE under a specific noise/label distribution. Squared loss = Gaussian noise (regression). Cross-entropy = Bernoulli (binary classification) or Categorical (multiclass). This isn't a coincidence — the MLE framework is *how* these loss functions were derived historically. Knowing this tells you which loss function to use for a new problem: model the distribution of your labels, take the negative log-likelihood, and that's your loss. It also tells you what assumptions your loss function makes, so you know when it might fail.

---

## External Resources

| Resource | What it covers | When to use |
|---|---|---|
| [CS229 Notes 1](https://cs229.stanford.edu/notes/cs229-notes1.pdf) | Full MLE derivation for both regression and classification | Primary reference for this lecture |
| [3Blue1Brown — What is backpropagation?](https://www.youtube.com/watch?v=Ilg3gGewQ5U) | Visual sigmoid + cross-entropy | Before neural networks |
| [StatQuest — MLE clearly explained](https://www.youtube.com/watch?v=XepXtl9YKwc) | Best visual MLE intro | MLE intuition |
| [StatQuest — Logistic Regression](https://www.youtube.com/watch?v=yIYKR4sgzI8) | Step-by-step logistic regression | After this notebook |
| [Understanding Deep Learning — Prince Ch. 5](https://udlbook.github.io/udlbook/) | MLE framework in modern context | Deeper treatment |
| [Newton's Method visualization](https://mathlets.org/mathlets/newtons-method/) | Interactive Newton's method | Geometric intuition |